# Standalone Google Colab Pipeline Parameter & Latency Benchmarker

This notebook allows you to test and determine the **optimal concurrency parameters (`max_workers`, `batch_size`)** and verify stagewise latencies when running the AWS Indian Judgments pipeline on Google Colab.

### Benchmark Goals:
1. **Worker Scaling**: Compare throughput (PDFs/sec & pages/sec) across worker pool sizes (`max_workers = 4, 8, 16, 32`).
2. **Connection Pooling**: Measure latency reduction from HTTP session connection reuse.
3. **Stagewise Latency Profiling**: Measure exact duration for `download_ms`, `extract_ms`, `entity_ms`, and `metadata_ms` in Colab.
4. **Optimal Parameter Recommendation**: Output the best parameter combination for full High Court dataset ingestion.

## Step 1: Install Dependencies

In [ ]:
!pip install -q pymupdf pyarrow duckdb rich spacy requests urllib3 matplotlib psutil

## Step 2: Auto-Load Pipeline Code & Modules
Clones repository automatically into Colab VM `/content/aws_indian_judgements` and adds it to `sys.path` so `from config import PipelineConfig` works out of the box.

In [ ]:
import os, sys
repo_dir = "/content/aws_indian_judgements"
if os.path.exists("/content") and not os.path.exists(repo_dir):
    print("Cloning repository into Colab environment...")
    !git clone https://github.com/duttadev/aws_indian_judgements.git {repo_dir}
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)
elif os.path.exists(repo_dir):
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)

print(f"Working Directory: {os.getcwd()}")
from config import PipelineConfig
from pipeline.storage import StorageManager
from batch_runner import BatchScheduler
print("✓ Pipeline modules imported successfully!")

## Step 3: Define Sample Benchmark Keys

In [ ]:
import time, json, glob
import pandas as pd
import matplotlib.pyplot as plt

# Load or generate benchmark S3 keys
sample_keys_file = "./data/sc_keys_sample.json"
if os.path.exists(sample_keys_file):
    with open(sample_keys_file, "r") as f:
        benchmark_keys = json.load(f)[:100]
else:
    benchmark_keys = [f"judgments/1950/doc_{i}.pdf" for i in range(1, 101)]

print(f"Loaded {len(benchmark_keys)} benchmark PDF keys.")

## Step 4: Run Worker Scaling Benchmark (`max_workers = 4, 8, 16, 32`)

In [ ]:
worker_options = [4, 8, 16, 32]
worker_benchmark_results = []

for workers in worker_options:
    print(f"\n==================================================")
    print(f" Testing max_workers = {workers} (Batch Size = 50, Sample N = {len(benchmark_keys)})")
    print(f"==================================================")
    
    scratch_path = f"/tmp/benchmark_workers_{workers}"
    os.makedirs(scratch_path, exist_ok=True)
    
    config = PipelineConfig(
        run_id=f"bench-workers-{workers}",
        base_output_dir=scratch_path,
        local_scratch_dir=scratch_path,
        batch_size=50,
        max_workers=workers,
        keep_pdf_files=False,
        keep_intermediate_artifacts=True,
        resume_enabled=False
    )
    
    storage = StorageManager(config)
    scheduler = BatchScheduler(config, storage)
    
    t0 = time.time()
    summary = scheduler.run_batch_pipeline(benchmark_keys)
    elapsed = time.time() - t0
    
    metrics_file = os.path.join(config.benchmarks_dir, f"metrics_{config.run_id}.jsonl")
    avg_download, avg_extract, avg_entity, avg_total = 0, 0, 0, 0
    if os.path.exists(metrics_file):
        records = []
        with open(metrics_file, "r") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
        if records:
            avg_download = sum(r.get("download_ms", 0) for r in records) / len(records)
            avg_extract = sum(r.get("extract_ms", 0) for r in records) / len(records)
            avg_entity = sum(r.get("entity_ms", 0) for r in records) / len(records)
            avg_total = sum(r.get("total_ms", 0) for r in records) / len(records)
            
    worker_benchmark_results.append({
        "max_workers": workers,
        "elapsed_sec": round(elapsed, 2),
        "pdfs_per_sec": summary.get("throughput_pdfs_per_sec", 0),
        "pages_per_sec": summary.get("throughput_pages_per_sec", 0),
        "avg_download_ms": round(avg_download, 1),
        "avg_extract_ms": round(avg_extract, 1),
        "avg_entity_ms": round(avg_entity, 1),
        "avg_total_ms": round(avg_total, 1),
        "peak_ram_mb": summary.get("peak_ram_mb", 0)
    })

df_workers = pd.DataFrame(worker_benchmark_results)
print("\n=== Worker Scaling Benchmark Summary ===")
print(df_workers.to_string(index=False))

## Step 5: Visualize Worker Scaling & Latency Breakdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_workers["max_workers"], df_workers["pdfs_per_sec"], marker="o", color="#1f77b4", linewidth=2.5)
ax1.set_title("Throughput (PDFs / sec) vs Worker Threads", fontweight="bold")
ax1.set_xlabel("max_workers")
ax1.set_ylabel("PDFs / sec")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.bar(df_workers["max_workers"].astype(str), df_workers["avg_download_ms"], label="Download (ms)", color="#2ca02c")
ax2.bar(df_workers["max_workers"].astype(str), df_workers["avg_extract_ms"], bottom=df_workers["avg_download_ms"], label="Extract (ms)", color="#ff7f0e")
ax2.bar(df_workers["max_workers"].astype(str), df_workers["avg_entity_ms"], bottom=df_workers["avg_download_ms"]+df_workers["avg_extract_ms"], label="Entity (ms)", color="#d62728")
ax2.set_title("Stagewise Latency Breakdown (ms / doc)", fontweight="bold")
ax2.set_xlabel("max_workers")
ax2.set_ylabel("Duration (ms)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Step 6: Parameter Recommendation

In [ ]:
best_row = df_workers.loc[df_workers["pdfs_per_sec"].idxmax()]
print(f"============================================================")
print(f" RECOMMENDED GOOGLE COLAB CONFIGURATION:")
print(f" • Optimal max_workers : {int(best_row["max_workers"])}")
print(f" • Peak Throughput    : {best_row["pdfs_per_sec"]} PDFs/sec ({best_row["pages_per_sec"]} pages/sec)")
print(f" • Avg Download Latency: {best_row["avg_download_ms"]} ms")
print(f" • Peak RAM RSS Memory : {best_row["peak_ram_mb"]} MB")
print(f"============================================================")